# Practical Exercise: Understanding Maximum Likelihood Estimation (MLE) and Maximum A Posteriori (MAP)

Welcome to the practical exercise on MLE and MAP. In this exercise, we will learn how to apply these two important estimation methods in a Linear Regression problem combined with Basis Functions.

## Learning Objectives:
1. Understand the meaning of MLE and its equivalence to the Least Squares method.
2. Understand the meaning of MAP and the incorporation of Prior knowledge into the learning process.
3. Observe the Overfitting phenomenon with MLE when using complex models and how MAP (with L2 Regularization) helps overcome this.

---
## Theory Recap:
- **MLE (Maximum Likelihood Estimation):** Find the weight vector $\mathbf{w}$ that maximizes the probability of observing the given data. In regression with Gaussian noise, MLE is equivalent to minimizing the sum of squared errors (Least Squares).
  Formula: $\mathbf{w}_{MLE} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$

- **MAP (Maximum A Posteriori):** Find the weight vector $\mathbf{w}$ that maximizes the posterior probability $P(\mathbf{w}|D)$. It combines MLE with a prior distribution of $\mathbf{w}$. If we assume $\mathbf{w}$ follows a Gaussian distribution (with variance controlled by parameter $\alpha$), MAP is equivalent to Ridge regression (L2 Regularization).
  Formula: $\mathbf{w}_{MAP} = (\mathbf{X}^T\mathbf{X} + \frac{\alpha}{\beta}\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$
  Where $\beta = 1/\sigma^2$ (noise precision), and $\alpha$ controls the strength of the Prior.

Let's get started!

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Set default figure size
plt.rcParams['figure.figsize'] = (10, 6)


---
## Part 1: Data Generation

First, we will create a synthetic dataset. Assume the underlying true data follows a non-linear function: $y = \sin(x^2 + 1)$.
However, in reality, collected data always contains noise. We will simulate this by adding Gaussian noise to the $y$ values.

**Requirement 1:**
1. Generate 50 data points in the interval $[0, 1]$ for the variable $x$.
2. Calculate the variable $y$ according to the function $\sin(x^2 + 1)$.
3. Add Gaussian noise with mean $\mu = 0$ and standard deviation $\sigma = 0.05$ to $y$ to generate the observed values $t$.
4. Plot a graph displaying the ground truth curve (red color) and the noisy data points (green scatter plot).

In [2]:
# Set seed for reproducible results
np.random.seed(42)

# --- WRITE YOUR CODE HERE ---
# 1. Create variable x from 0 to 1, step 0.02 (total 50 points)

# 2. Generate the ground truth curve y = sin(x^2 + 1)

# 3. Add Gaussian noise with mean=0, std=0.05

# 4. Plot the graph (red line for ground truth, green scatter for noisy data)


---
## Part 2: Building the Feature Matrix

Since the underlying function is non-linear, we cannot just use 1st-degree $x$ for prediction. Instead, we will use polynomials of degree $M-1$ as Basis Functions.
The Design Matrix $\mathbf{X}$ with dimensions $N \times M$ will take the form:
$X_{i, j} = (x_i)^j$ where $i = 0 \dots N-1$ and $j = 0 \dots M-1$.

**Requirement 2:** Complete the `create_X` function below to build the design matrix.

In [3]:
def create_X(x, M):
    """
    Function to create the design matrix X using polynomials of degree M-1.
    Input:
      - x: 1D numpy array containing x values.
      - M: Number of parameters w (equivalent to max polynomial degree M-1).
    Output:
      - X: 2D matrix of shape (N, M)
    """
    N = len(x)
    X = np.zeros((N, M))
    
    # --- WRITE YOUR CODE HERE ---
    # Hint: Use a for loop from 0 to M-1, set X[:, j] = x ** j
    
    return X

# Test the function:
X_test = create_X(np.array([1, 2, 3]), M=3)
print("X test matrix:\n", X_test)


---
## Part 3: Estimation using MLE (Equivalent to Least Squares)

Now, we will use MLE to find the weight vector $\mathbf{w}$. The closed-form solution is:
$$\mathbf{w}_{MLE} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{t}$$

**Requirement 3:** 
1. Implement the `MLE_fit` function based on the formula above. (Use `np.linalg.inv` and `np.dot` or `@`).
2. Use a polynomial of degree 9 ($M=10$).
3. Plot a graph displaying the predicted results (blue line) alongside the data.

In [4]:
def MLE_fit(X, t):
    """
    Function to compute weights w using MLE.
    """
    # --- WRITE YOUR CODE HERE ---
    # Formula: w_mle = (X^T X)^{-1} X^T t
    # Hint: use np.linalg.inv and @ operator
    pass

# 1. Create X matrix for M = 10
M = 10
X_mle = create_X(x, M)

# 2. Find w
w_mle = MLE_fit(X_mle, t)

# 3. Predict y
y_mle = X_mle @ w_mle

# 4. Plot the graph
plt.plot(x, y_true, 'r-', label='Ground Truth Curve')
plt.plot(x, t, 'go', label='Noisy Data')
plt.plot(x, y_mle, 'b-', label='MLE Prediction (M=10)')
plt.legend()
plt.title('Prediction using MLE with M=10')
plt.show()

print("Observation: With M=10, the MLE model suffers from Overfitting. The curve tries to closely fit the noisy points.")


---
## Part 4: Estimation using MAP (Combining Prior - L2 Regularization)

As seen in the previous part, with $M=10$, the model is too complex and MLE leads to **Overfitting**. The coefficients of $\mathbf{w}$ can become very large.

To overcome this, we use MAP. By assuming $\mathbf{w}$ has a normal distribution around 0, we penalize these weights from becoming too large. The closed-form solution for MAP is:
$$\mathbf{w}_{MAP} = (\mathbf{X}^T\mathbf{X} + c\mathbf{I})^{-1}\mathbf{X}^T\mathbf{t}$$
Where $c = \frac{\alpha}{\beta}$ and $\mathbf{I}$ is the identity matrix. The parameter $c$ (acting as a regularization constant) penalizes large weights more heavily as it increases.

**Requirement 4:**
1. Implement the `MAP_fit` function.
2. Apply `MAP_fit` with $M=10$ and $c = 0.005$.
3. Plot a comparison graph.

In [5]:
def MAP_fit(X, t, c):
    """
    Function to compute weights w using MAP.
    Input:
      - c: ratio of alpha / beta
    """
    M = X.shape[1]
    # --- WRITE YOUR CODE HERE ---
    # Formula: w_map = (X^T X + c*I)^{-1} X^T t
    # Hint: use np.eye(M) to create identity matrix
    pass

# 1. Find w with MAP
c_value = 0.005
w_map = MAP_fit(X_mle, t, c_value)

# 2. Predict y
y_map = X_mle @ w_map

# 3. Plot comparison graph
plt.plot(x, y_true, 'r-', label='Ground Truth Curve')
plt.plot(x, t, 'go', label='Noisy Data')
plt.plot(x, y_mle, 'b--', alpha=0.5, label='MLE Prediction (M=10, Overfitted)')
plt.plot(x, y_map, 'm-', linewidth=2, label=f'MAP Prediction (M=10, c={c_value})')
plt.legend()
plt.title('MLE vs MAP Comparison')
plt.show()

print("Observation: Using MAP helps the curve to be smoother and prevents Overfitting even when using degree M=10.")


---
## Part 5: Error Calculation and Parameter Exploration

Visual evaluation is not enough. Let's write a function to calculate the Sum of Squared Errors (SSE) to measure the prediction accuracy compared to the actual data.
$E = \frac{1}{2} \sum_{i=1}^{N} (y_{pred}^{(i)} - t^{(i)})^2$

**Requirement 5:**
1. Write the function `compute_error(y_pred, t)`.
2. Print out the errors for the MLE model and MAP model for comparison.

In [6]:
def compute_error(y_pred, t):
    # --- WRITE YOUR CODE HERE ---
    # Formula: E = 0.5 * sum((y_pred - t)^2)
    pass

err_mle = compute_error(y_mle, t)
err_map = compute_error(y_map, t)

print(f"Error of MLE (M=10): {err_mle:.4f}")
print(f"Error of MAP (M=10, c=0.005): {err_map:.4f}")


### Extra Exercises:
Try changing the values of `M` (e.g., $M=2, 5, 10, 15$) and the value of $c$ (e.g., $c=0, 0.001, 0.1, 1$) to see how the graphs change! (Note: when $c=0$, MAP becomes MLE).

In [7]:
# Your experimental code:

